## Integration Modularization

This notebook captures the production-style integration flow now implemented under `src/`.

### What is now modularized

- `src/acu/config.py`: environment-backed settings model (`Settings`, `load_settings`)
- `src/acu/analyze.py`: ACU document analysis wrapper
- `src/acu/parse.py`: ACU field extraction + flattening helpers
- `src/storage/storage.py`: blob storage abstraction + stage enum (`RAW`, `ACU`, `ANNOTATED`)
- `src/integration/pipeline.py`: end-to-end orchestration (`run_pipeline`)
- `src/scripts/run_pipeline.py`: CLI runner for terminal usage


### 1) Setup imports and configuration

This cell loads `.env`, imports the modular components, and prints the effective ACU analyzer/config.


In [ ]:
from pathlib import Path
import sys
from dotenv import load_dotenv

# Ensure repo root is on import path when running from notebooks
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv()

from src.acu.config import load_settings

settings = load_settings()
print("ACU endpoint:", settings.azure_ai_endpoint)
print("ACU analyzer:", settings.acu_analyzer_id)
print("ACU api_version:", settings.acu_api_version)


### 2) Run the integrated pipeline

This executes the orchestrator in `src/integration/pipeline.py`:

1. Upload RAW PDF to blob (`raw`)
2. Call ACU analyzer
3. Save ACU output to blob (`acu`)
4. Return flattened extracted fields


In [ ]:
import json
from src.integration.pipeline import run_pipeline

sample_pdf = "data/AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf"
doc_id = "integration-demo-001"

result = run_pipeline(
    local_pdf_path=sample_pdf,
    settings=settings,
    document_id=doc_id,
)

print(json.dumps(result, indent=2)[:3000])


### 3) Inspect extracted fields

`run_pipeline(...)` returns flattened fields from ACU output for quick checks.


In [ ]:
fields = result.get("extracted_fields", {})
print("Total fields:", len(fields))
for k in sorted(list(fields.keys()))[:12]:
    print(f"- {k}: {str(fields[k])[:120]}")


### 4) Verify stored artifacts in Blob

Use the storage abstraction to fetch the persisted ACU JSON wrapper for this document id.


In [ ]:
from src.storage.storage import get_storage, Stage

storage = get_storage()
acu_bytes = storage.download_blob(doc_id, Stage.ACU, ".json")

if not acu_bytes:
    raise RuntimeError("No ACU blob found for document id")

acu_wrapper = json.loads(acu_bytes.decode("utf-8"))
print("Wrapper keys:", acu_wrapper.keys())
print("Metadata:", acu_wrapper.get("metadata", {}))
print("Top-level ACU keys:", list(acu_wrapper["data"].keys()))
